# 🃏 Poker Arena — Evidência Científica #1 (célula única)

**Rode a única célula abaixo** (`Shift+Enter`). Ela instala o OpenSpiel, treina **CFR** no **Kuhn** e no **Leduc**, plota a **exploitability caindo a ~0** (prova de convergência ao equilíbrio de Nash) e imprime a política aprendida.

Roda em **CPU** no Colab, em poucos minutos. Não precisa de GPU.

In [ ]:
# ============================================================
# Poker Arena - Evidencia Cientifica #1 (CELULA UNICA: rode tudo aqui)
# CFR converge ao equilibrio de Nash no Kuhn Poker (+ Leduc de bonus).
# Prova: exploitability -> 0. (Se a 1a execucao falhar em 'import pyspiel',
# rode a celula de novo: o Colab acabou de instalar o pacote.)
# ============================================================

# 1) Instala OpenSpiel (DeepMind, Apache-2.0)
!pip install -q open_spiel

# 2) Imports
import pyspiel
from open_spiel.python.algorithms import cfr, exploitability
import matplotlib.pyplot as plt

# 3) Treina CFR e mede a exploitability ao longo das iteracoes
def run_cfr(game_name, n_iters, every=5):
    game = pyspiel.load_game(game_name)
    solver = cfr.CFRSolver(game)
    iters, conv = [], []
    for i in range(1, n_iters + 1):
        solver.evaluate_and_update_policy()
        if i == 1 or i % every == 0:
            e = exploitability.exploitability(game, solver.average_policy())
            iters.append(i)
            conv.append(e)
            print(f'[{game_name}] iter {i:4d}  exploitability = {e:.6f}')
    return solver, iters, conv

kuhn_solver,  k_it, k_ex = run_cfr('kuhn_poker', 200)
leduc_solver, l_it, l_ex = run_cfr('leduc_poker', 100)

# 4) Graficos lado a lado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.semilogy(k_it, k_ex, marker='o', color='#c0392b')
ax1.set_title('CFR converge ao Nash - Kuhn Poker')
ax1.set_xlabel('Iteracao de CFR'); ax1.set_ylabel('Exploitability (log)')
ax1.grid(True, which='both', alpha=0.3)
ax2.semilogy(l_it, l_ex, marker='o', color='#2980b9')
ax2.set_title('CFR em Leduc Holdem')
ax2.set_xlabel('Iteracao de CFR'); ax2.set_ylabel('Exploitability (log)')
ax2.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('cfr_convergence.png', dpi=150)
plt.show()

# 5) Politica media aprendida no Kuhn (compare com o Nash teorico)
print()
print('=== Politica media aprendida (Kuhn) ===')
print('info_state -> [P(pass/check/fold), P(bet/call)]')
avg = kuhn_solver.average_policy()
for info_state, idx in sorted(avg.state_lookup.items()):
    print(f'  {info_state:6s} -> {avg.action_probability_array[idx].round(3)}')

print()
print(f'Exploitability final  Kuhn : {k_ex[-1]:.6f}   (deve ~0 = otimo)')
print(f'Exploitability final  Leduc: {l_ex[-1]:.6f}')
print('Graficos salvos em cfr_convergence.png')